# LightGBM + Optuna notebook example

这个 notebook 演示如何在 Jupyter / VS Code Notebook 中直接调用函数版接口进行调参。

主要接口：

- `run_lightgbm_optuna(train_df=..., valid_df=..., oot_df=..., ...)`
- `run_lightgbm_optuna_from_df(df=..., target_col=..., split_col='split', ...)`

进度可视化说明：

- `show_progress=True`：显示每个 phase 的 trial 进度
- `show_progress=False`：关闭 trial 级进度，只保留必要输出
- 安装了 `tqdm` 时会显示 notebook / 终端进度条；否则自动退化成文本进度


In [ ]:
import numpy as np
import pandas as pd

from lightgbm_optuna_tuner import run_lightgbm_optuna_from_df

In [ ]:
rng = np.random.default_rng(42)

def make_df(n: int, split: str) -> pd.DataFrame:
    age = rng.integers(18, 61, size=n)
    income = rng.normal(12000, 3500, size=n).clip(1000, None)
    city = rng.choice(['a', 'b', 'c', 'd'], size=n)
    channel = rng.choice(['app', 'web', 'offline'], size=n, p=[0.55, 0.35, 0.10])
    score = 0.035 * (age - 35) + 0.00012 * (income - 12000)
    score += (city == 'd') * 0.45 + (channel == 'offline') * 0.35 + (channel == 'app') * 0.15
    prob = 1.0 / (1.0 + np.exp(-score))
    target = rng.binomial(1, prob)
    return pd.DataFrame(
        {
            'split': split,
            'age': age,
            'income': income.round(2),
            'city': city,
            'channel': channel,
            'target': target,
            'user_id': np.arange(n),
        }
    )

df = pd.concat(
    [
        make_df(1200, 'train'),
        make_df(400, 'test'),
        make_df(400, 'oot'),
    ],
    ignore_index=True,
)
df.head()

In [ ]:
result = run_lightgbm_optuna_from_df(
    df,
    target_col='target',
    split_col='split',
    task='binary',
    drop_cols=['user_id'],
    categorical_cols=['city', 'channel'],
    fast_phase_trials=2,
    main_phase_trials=3,
    fast_phase_train_rows=500,
    fast_phase_valid_rows=200,
    max_tune_train_rows=800,
    max_tune_valid_rows=300,
    num_threads=4,
    trial_parallelism=1,
    save_artifacts=False,
    verbose=True,
    show_progress=True,
)


In [ ]:
result.best_params

In [ ]:
result.summary

In [ ]:
result.feature_cols, result.categorical_cols